In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from collections import OrderedDict

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Find the file path (click the folder icon on the left → browse to your file)
# Example path (change this to yours):
# /content/drive/MyDrive/YourFolder/your_file.csv

# Step 3: Read the file with pandas
# Replace this with your actual file path
file_path = '/content/drive/MyDrive/Colab Notebooks/options_SPX_final_minimal_scope.csv'    # ← CHANGE THIS
date_col = "date"

usecols = [
    "date", "cp_flag", "moneyness", "time_to_maturity", "impl_volatility"
]

dtypes = {
    "cp_flag": "category",
    "moneyness": "float64",
    "time_to_maturity": "float64",
    "impl_volatility": "float64",
}

# Read the whole file once – 1.3GB is usually fine on a modern machine.
df = pd.read_csv(
    file_path,
    usecols=usecols,
    dtype=dtypes,
    parse_dates=[date_col],
    low_memory=False,
)

# Out-of-the-money filter: calls with m > 1, puts with m < 1
is_otm = ((df["cp_flag"] == "C") & (df["moneyness"] > 1.0)) | \
         ((df["cp_flag"] == "P") & (df["moneyness"] < 1.0))
df = df[is_otm]

# Moneyness and TTM ranges (paper uses m in [0.5,1.5], τ between about 1 month and 1 year)
df = df[(df["moneyness"].between(0.5, 1.2)) &
        (df["time_to_maturity"].between(28/365, 1.0))]


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# @title
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
# Alternative: 3D surface plot with aggregated counts
# Create 2D histogram for surface plot
moneyness_bins = np.linspace(0.5, 1.2, 20)
ttm_bins = np.linspace(28/365, 1.0, 20)

hist, xedges, yedges = np.histogram2d(
    df['moneyness'],
    df['time_to_maturity'],
    bins=[moneyness_bins, ttm_bins]
)

# Create meshgrid for surface plot
X, Y = np.meshgrid(xedges[:-1], yedges[:-1])
Z = hist.T

# Create 3D surface plot
fig = go.Figure(data=[go.Surface(z=Z, x=X, y=Y, colorscale='Viridis')])

fig.update_layout(
    title='3D Surface: Options Data Distribution by Moneyness and TTM',
    scene=dict(
        xaxis_title='Moneyness (K/S)',
        yaxis_title='Time to Maturity (years)',
        zaxis_title='Number of Options',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))
    ),
    width=800,
    height=600
)

fig.show()

In [ ]:
# Regular grid in m and tau
m_grid = np.linspace(0.5, 1.2, 21)           # 21 points in moneyness
tau_grid = np.linspace(28/365, 1.0, 20)      # 20 maturities

M, T = np.meshgrid(m_grid, tau_grid, indexing="xy")
grid_points = np.column_stack([M.ravel(), T.ravel()])  # (G, 2)
G = grid_points.shape[0]

# Kernel bandwidths (you can tune these – start with something reasonable)
h_m = 0.05
h_tau = 0.05

def smooth_surface_for_day(day_df):
    """
    Nadaraya–Watson 2D Gaussian smoother for one trading day.
    day_df has columns: moneyness, time_to_maturity, impl_volatility
    Returns a (len(tau_grid), len(m_grid)) array.
    """
    m_i = day_df["moneyness"].values          # (N,)
    tau_i = day_df["time_to_maturity"].values # (N,)
    iv_i = day_df["impl_volatility"].values   # (N,)

    if len(day_df) == 0:
        return np.full((len(tau_grid), len(m_grid)), np.nan)

    # Differences grid - data (broadcasting)
    dm = grid_points[:, 0][:, None] - m_i[None, :]
    dt = grid_points[:, 1][:, None] - tau_i[None, :]

    # Gaussian kernel weights
    w = np.exp(-0.5 * ((dm / h_m) ** 2 + (dt / h_tau) ** 2))

    num = w @ iv_i                 # (G,)
    den = w.sum(axis=1)            # (G,)

    iv_grid_flat = num / den
    iv_grid = iv_grid_flat.reshape(len(tau_grid), len(m_grid))

    return iv_grid


In [ ]:
# @title
# Pick a specific day (choose one that has data)
sample_date = df['date'].dt.date.unique()[0]  # First available day
print(f"Plotting data for: {sample_date}")

# Filter data for the selected day
day_data = df[df['date'].dt.date == sample_date]

# Create 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=day_data['moneyness'],
    y=day_data['time_to_maturity'],
    z=day_data['impl_volatility'],
    mode='markers',
    marker=dict(
        size=3,
        color=day_data['impl_volatility'],
        colorscale='Viridis',
        opacity=0.7
    )
)])

fig.update_layout(
    title=f'Options IV Surface - {sample_date}',
    scene=dict(
        xaxis_title='Moneyness (K/S)',
        yaxis_title='Time to Maturity (years)',
        zaxis_title='Implied Volatility'
    ),
    width=800,
    height=600
)

fig.show()

Plotting data for: 2015-01-02


In [ ]:
# @title
# Pick a specific day (choose one that has data)
sample_date = df['date'].dt.date.unique()[0]  # First available day
print(f"Plotting data for: {sample_date}")

# Filter data for the selected day
day_data = df[df['date'].dt.date == sample_date]

# Fit the surface using your function
fitted_surface = smooth_surface_for_day(day_data)

# Create 3D plot with both scatter points and surface
fig = go.Figure()

# Add scatter plot of data points
fig.add_trace(go.Scatter3d(
    x=day_data['moneyness'],
    y=day_data['time_to_maturity'],
    z=day_data['impl_volatility'],
    mode='markers',
    marker=dict(
        size=3,
        color=day_data['impl_volatility'],
        colorscale='Viridis',
        opacity=0.7
    ),
    name='Data Points'
))

# Add surface plot of fitted surface
fig.add_trace(go.Surface(
    x=m_grid,
    y=tau_grid,
    z=fitted_surface,
    colorscale='Plasma',
    opacity=0.7,
    name='Fitted Surface'
))

fig.update_layout(
    title=f'Options IV Surface with Fitted Smooth Surface - {sample_date}',
    scene=dict(
        xaxis_title='Moneyness (K/S)',
        yaxis_title='Time to Maturity (years)',
        zaxis_title='Implied Volatility'
    ),
    width=800,
    height=600
)

fig.show()

Plotting data for: 2015-01-02


In [ ]:
# Sort by date just in case
df = df.sort_values(date_col)

daily_surfaces = OrderedDict()  # date -> 2D array (tau x m)

for d, day_df in df.groupby(date_col):
    surf = smooth_surface_for_day(day_df)
    # Use log implied vol, as they do (better behaved and matches their KL step)
    daily_surfaces[d] = np.log(surf)

dates = np.array(list(daily_surfaces.keys()))
surfaces = np.stack(list(daily_surfaces.values()), axis=0)  # (T, Ntau, Nm)
T_obs = surfaces.shape[0]


In [ ]:
# ΔX_t shape: (T-1, Ntau, Nm)
delta_surfaces = surfaces[1:, :, :] - surfaces[:-1, :, :]
delta_dates = dates[1:]

# Flatten each surface to a vector (features = grid points)
delta_matrix = delta_surfaces.reshape(delta_surfaces.shape[0], -1)  # (T-1, G)


In [ ]:
delta_matrix

array([[ 0.01643473,  0.01364837,  0.03869003, ...,  0.08888213,
         0.07489494,  0.0685783 ],
       [ 0.01920705,  0.02903099,  0.02873232, ..., -0.03535028,
         0.01894478,  0.02301883],
       [ 0.04762059,  0.01312498, -0.02281534, ...,  0.0034655 ,
        -0.03909185, -0.04259932],
       ...,
       [ 0.05570327,  0.05147811,  0.02842396, ..., -0.04192951,
        -0.04842562, -0.05484482],
       [ 0.01576071,  0.0120948 ,  0.01133704, ...,  0.02471777,
         0.02591247,  0.03289113],
       [ 0.02127877,  0.02342334,  0.01245529, ..., -0.00256927,
         0.00073587,  0.0050123 ]])

In [ ]:
# Keep first few factors; paper finds 2–3 explain most of the variance
n_factors = 5
pca = PCA(n_components=n_factors)
pca.fit(delta_matrix)

eigenvalues = pca.explained_variance_            # ν_k^2
explained_ratio = pca.explained_variance_ratio_  # proportion of variance
eigenmodes_flat = pca.components_                # shape (n_factors, G)

# Reshape eigenmodes back to (tau, m) surfaces
eigenmodes = eigenmodes_flat.reshape(n_factors, len(tau_grid), len(m_grid))

# Factor time series: projections x_k(t)
factor_scores = pca.transform(delta_matrix)      # shape (T-1, n_factors)


ValueError: Input X contains NaN.
PCA does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
np.save("spx_m_grid.npy", m_grid)
np.save("spx_tau_grid.npy", tau_grid)
np.save("spx_eigenmodes.npy", eigenmodes)
np.save("spx_factor_scores.npy", factor_scores)
np.save("spx_eigenvalues.npy", eigenvalues)
np.save("spx_explained_ratio.npy", explained_ratio)
